# Step 0 · [1] Candidate pool

Applies the selection rules of `config/sector.yaml` and returns **one row per DOCDB family** with a 0/1 flag per rule:

| Flag | Rule |
|---|---|
| `r_core_cpc`, `r_core_ipc`, `r_keyword` | **A** · core codes or specific keywords: in the sector on their own |
| `r_component_cpc`, `r_component_ipc` | **B** · generic codes **and** an eyewear term in the family |
| `r_eyewear_cpc`, `r_eyewear_ipc` | **C** · spectacle codes **and** an electronic term in the family |
| `r_review` | head-up / head-mounted display codes: candidates for the semantic filter (step 3) |
| `r_excluded` | an exclusion term appears in the family |

`in_sector` = A or B or C, and not excluded. Families with only `r_review` stay out for now (`review_only`).

Rules as in the tutorials: DOCDB families, `ipr_type = 'PI'`, no dummy applications, earliest priority year in the window. Everything is computed in the database; only the flags are downloaded.

Run on **TEST** first, then **PROD**.

In [ ]:
%pip install -q pyyaml

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from epo.tipdata.patstat import PatstatClient

from src.sector import SECTOR_RULES, candidate_pool_query, family_titles, load_config

ENV = "TEST"                       # "TEST" to check the code, "PROD" for the real numbers
OUTPUT_DIR = Path("outputs/step0")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 100)

cfg = load_config()
patstat = PatstatClient(env=ENV)
db = patstat.orm()

YEAR_MIN, YEAR_MAX = cfg["years"]["min"], cfg["years"]["max"]
LAST_COMPLETE_YEAR = YEAR_MAX - cfg["years"]["truncated"]
print(f"{cfg['sector_name']} | env {ENV} | years {YEAR_MIN}-{YEAR_MAX} | complete up to {LAST_COMPLETE_YEAR}")

## 1. The candidate pool

In [ ]:
cand = patstat.df(candidate_pool_query(db, cfg))

rule_cols = [c for c in cand.columns if c.startswith("r_")]
cand[rule_cols] = cand[rule_cols].astype(int)
cand["in_sector"] = (cand[list(SECTOR_RULES)].max(axis=1) == 1) & (cand["r_excluded"] == 0)
cand["review_only"] = (cand["r_review"] == 1) & (cand[list(SECTOR_RULES)].max(axis=1) == 0) & (cand["r_excluded"] == 0)
cand["n_rules"] = cand[list(SECTOR_RULES)].sum(axis=1)
cand["truncated"] = cand["year"] > LAST_COMPLETE_YEAR

pd.Series({
    "candidate families": len(cand),
    "in sector (A or B or C, not excluded)": int(cand["in_sector"].sum()),
    "review only (HUD/HMD, no eyewear term)": int(cand["review_only"].sum()),
    "excluded by exclude_terms": int(((cand[list(SECTOR_RULES)].max(axis=1) == 1) & (cand["r_excluded"] == 1)).sum()),
}, name="families").to_frame()

## 2. Contribution of each rule

As in Tutorial 0, Section 5.6: how many sector families each rule finds, and how many **only** that rule finds (its unique contribution).

In [ ]:
sector = cand[cand["in_sector"]]
contribution = pd.DataFrame({
    "families": sector[list(SECTOR_RULES)].sum(),
    "only this rule": sector[sector["n_rules"] == 1][list(SECTOR_RULES)].sum(),
})
contribution["share of sector (%)"] = (contribution["families"] / len(sector) * 100).round(1)
contribution.loc["sector (union)"] = [len(sector), None, 100.0]
contribution

## 3. Families per priority year

In [ ]:
per_year = (
    pd.concat([cand[cand["in_sector"]].assign(group="in sector"),
               cand[cand["review_only"]].assign(group="review only")])
    .groupby(["year", "group"]).size().unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 4.5))
per_year.plot(ax=ax, marker="o", ms=3)
ax.axvspan(LAST_COMPLETE_YEAR + 0.5, YEAR_MAX + 0.5, color="grey", alpha=0.15, lw=0, label="incomplete years")
ax.set(title=f"{cfg['sector_name']}: candidate families per earliest priority year ({ENV})",
       xlabel="earliest priority year", ylabel="families", xlim=(YEAR_MIN, YEAR_MAX))
ax.legend()
plt.show()

## 4. Eyeballing: random titles

For each rule, families found **only** by that rule (the riskiest ones), plus review-only and excluded families. Read them: are they smart glasses?

In [ ]:
N_SAMPLE = 8


def one_title_per_family(titles):
    """Prefer an English title; otherwise any title, prefixed with its language."""
    lang = titles["appln_title_lg"].str.strip().str.lower()       # PATSTAT may store "EN" or "en"
    titles = titles.assign(is_en=lang == "en").sort_values("is_en", ascending=False)
    first = titles.drop_duplicates("docdb_family_id")
    return [t if en else f"[{lg}] {t}" for en, lg, t in zip(first["is_en"], first["appln_title_lg"], first["appln_title"])]


def show_sample(label, families):
    ids = families["docdb_family_id"].sample(min(N_SAMPLE, len(families)), random_state=42)
    if ids.empty:
        print(f"--- {label}: none")
        return
    titles = patstat.df(family_titles(db, ids))
    print(f"--- {label} ({len(families):,} families)")
    if titles.empty:
        print("    (no titles in PATSTAT for these families)")
        return
    for t in one_title_per_family(titles):
        print("   ", t)


only_one = sector[sector["n_rules"] == 1]
for rule in SECTOR_RULES:
    show_sample(f"only {rule}", only_one[only_one[rule] == 1])
show_sample("review only", cand[cand["review_only"]])
show_sample("excluded", cand[(cand["r_excluded"] == 1) & (cand[list(SECTOR_RULES)].max(axis=1) == 1)])

## 5. Save

The table stays on TIP (`outputs/` is git-ignored): the repository keeps the config and the queries that rebuild it.

In [ ]:
path = OUTPUT_DIR / f"candidates_{ENV}.parquet"
try:
    cand.to_parquet(path, index=False)
except ImportError:                       # no pyarrow: fall back to CSV
    path = path.with_suffix(".csv")
    cand.to_csv(path, index=False)
print(f"saved {len(cand):,} families to {path}")